In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Loading data
df = pd.read_csv('GUG_data.csv')

df_test = df.copy()

# Identify columns that have missing values
columns_with_missing = df_test.columns[df_test.isna().sum() > 0]

# Features used for the random forest model
features = ["Year", "Average Entry Tariff", "Student: staff ratio", "Expenditure per student (fte)", "Career prospects"]

# Save results
results = {}

for target_col in columns_with_missing:
    # Randomly remove 20% of the values ​​in this column to evaluate the methods
    np.random.seed(42)
    missing_indices = np.random.choice(df_test.index[df_test[target_col].notna()], size=int(0.2 * df_test[target_col].count()), replace=False)
    true_values = df_test.loc[missing_indices, target_col]
    df_test.loc[missing_indices, target_col] = np.nan  # مقدار را حذف می‌کنیم

    # Method 1: Filling with mean
    mean_imputer = SimpleImputer(strategy="mean")
    df_mean_filled = df_test.copy()
    df_mean_filled[target_col] = mean_imputer.fit_transform(df_mean_filled[[target_col]])

    # Method 2: Filling with university average
    df_university_filled = df_test.copy()
    df_university_filled[target_col] = df_university_filled.groupby("Institution")[target_col].transform(lambda x: x.fillna(x.mean()))
    df_university_filled[target_col] = df_university_filled[target_col].fillna(df_university_filled[target_col].mean())

    # Method 3: Filling with KNN Imputation
    knn_imputer = KNNImputer(n_neighbors=5)
    df_knn_filled = df_test.copy()
    df_knn_filled[target_col] = knn_imputer.fit_transform(df_knn_filled[[target_col]])

    # Method 4: Filling with Random Forest Model
    df_rf_filled = df_test.copy()

    train_df = df_rf_filled.dropna(subset=[target_col])
    test_df = df_rf_filled[df_rf_filled[target_col].isna()]

    X_train, y_train = train_df[features], train_df[target_col]
    X_test = test_df[features]

    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_train.mean())

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    df_rf_filled.loc[df_rf_filled[target_col].isna(), target_col] = model.predict(X_test)

    # Remove NaNs from true_values ​​and predicted data to prevent errors
    predicted_values_mean = df_mean_filled.loc[missing_indices, target_col].dropna()
    predicted_values_university = df_university_filled.loc[missing_indices, target_col].dropna()
    predicted_values_knn = df_knn_filled.loc[missing_indices, target_col].dropna()
    predicted_values_rf = df_rf_filled.loc[missing_indices, target_col].dropna()

    min_length = min(len(true_values), len(predicted_values_mean), len(predicted_values_university), len(predicted_values_knn), len(predicted_values_rf))
    true_values = true_values.iloc[:min_length]
    predicted_values_mean = predicted_values_mean.iloc[:min_length]
    predicted_values_university = predicted_values_university.iloc[:min_length]
    predicted_values_knn = predicted_values_knn.iloc[:min_length]
    predicted_values_rf = predicted_values_rf.iloc[:min_length]

    # Calculate MAE
    mae_mean = mean_absolute_error(true_values, predicted_values_mean)
    mae_university = mean_absolute_error(true_values, predicted_values_university)
    mae_knn = mean_absolute_error(true_values, predicted_values_knn)
    mae_rf = mean_absolute_error(true_values, predicted_values_rf)

    # Save results
    results[target_col] = {
        "Mean Imputation": mae_mean,
        "University Mean Imputation": mae_university,
        "KNN Imputation": mae_knn,
        "Random Forest Imputation": mae_rf,
        "Best Method": min([(mae_mean, "Mean Imputation"),
                             (mae_university, "University Mean Imputation"),
                             (mae_knn, "KNN Imputation"),
                             (mae_rf, "Random Forest Imputation")])[1]
    }

# Display results for each column
for col, res in results.items():
    print(f"Column: {col}")
    print(f"Mean Imputation MAE: {res['Mean Imputation']}")
    print(f"University Mean Imputation MAE: {res['University Mean Imputation']}")
    print(f"KNN Imputation MAE: {res['KNN Imputation']}")
    print(f"Random Forest Imputation MAE: {res['Random Forest Imputation']}")
    print(f"Best Method: {res['Best Method']}")
    print("-" * 50)


Column: % Satisfied with Teaching
Mean Imputation MAE: 5.737339399348587
University Mean Imputation MAE: 5.246132521729853
KNN Imputation MAE: 5.737339399348587
Random Forest Imputation MAE: 5.402160053333379
Best Method: University Mean Imputation
--------------------------------------------------
Column: % Satisfied with course
Mean Imputation MAE: 7.159045986742484
University Mean Imputation MAE: 6.55990828597198
KNN Imputation MAE: 7.159045986742484
Random Forest Imputation MAE: 6.833834742354398
Best Method: University Mean Imputation
--------------------------------------------------
Column: Continuation
Mean Imputation MAE: 3.821253216833012
University Mean Imputation MAE: 2.615555514231243
KNN Imputation MAE: 3.821253216833012
Random Forest Imputation MAE: 3.2501344563937686
Best Method: University Mean Imputation
--------------------------------------------------
Column: Expenditure per student (fte)
Mean Imputation MAE: 1.8961126870674516
University Mean Imputation MAE: 1.297